# 01b Parse MEPS Deterministic Forecast - Daily and Window Forecast Panels

This notebook converts archived deterministic MEPS NetCDF forecast trajectories into canonical forecast-weather panels. It preserves the existing UTC-day daily forecast output and adds Europe/Oslo local-time window outputs with a horizon-hour coverage audit.


## Imports and constants

The setup defines project-root discovery, raw MEPS paths, variable mappings, output locations, horizon filters, physical bounds, and local-time window specifications.


In [ ]:
import gc
import os
import re
import time
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import netCDF4
except ImportError as exc:
    raise ImportError(
        "netCDF4 is required for this parser. Install with `mamba install -n csvi_env netCDF4`."
    ) from exc

def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README_FOR_CODEX.md").exists() and (candidate / "AGENTS.md").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate project root from README_FOR_CODEX.md and AGENTS.md. "
        "Start Jupyter from the repository root."
    )


def resolve_raw_data_root() -> Path:
    candidates = []
    env_value = os.environ.get("FIMEX_OUT_ROOT")
    if env_value:
        candidates.append(Path(env_value))
    candidates.extend([
        Path("C:/Users/<USER>/fimex_out"),
        Path("/mnt/c/Users/<USER>/fimex_out"),
    ])
    for candidate in candidates:
        if candidate.exists():
            return candidate
    expected = " or ".join(str(p) for p in candidates)
    raise FileNotFoundError(f"Missing raw NetCDF root. Expected one of: {expected}")


PROJECT_ROOT = find_project_root()
RAW_DATA_ROOT = resolve_raw_data_root()
AVD_LOKASJON = PROJECT_ROOT / "thesis_context" / "Avd_lokasjon.xlsx"
CFG_PATH = RAW_DATA_ROOT / "cloud_points.cfg"
REALISED_DAILY_PATH = PROJECT_ROOT / "data" / "processed" / "realised_weather_daily.parquet"
REALISED_WINDOWS_PATH = PROJECT_ROOT / "data" / "processed" / "realised_weather_daily_windows.parquet"
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "forecast_meps_daily.parquet"
WINDOW_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "forecast_meps_daily_windows.parquet"
HOUR_AUDIT_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "forecast_meps_horizon_hour_audit.parquet"
LOCAL_TIMEZONE = "Europe/Oslo"

VARIANTS = {
    "temp":   ("temperature_nc",    "air_temperature_2m",       "temp",
               re.compile(r"^points_([0-9]{8})T([0-9]{2})Z_temp[.]nc$")),
    "precip": ("precipitation_nc",  "precipitation_amount_acc", "precip",
               re.compile(r"^points_([0-9]{8})T([0-9]{2})Z_precip[.]nc$")),
    "wind":   ("wind_nc",           "wind_speed",               "wind",
               re.compile(r"^points_([0-9]{8})T([0-9]{2})Z_wind[.]nc$")),
    "humid":  ("humidity_nc",       "relative_humidity_2m",     "humid",
               re.compile(r"^points_([0-9]{8})T([0-9]{2})Z_humid[.]nc$")),
    "cloud":  ("cloud_daily_nc",    "cloud_area_fraction",      "cloud",
               re.compile(r"^cloud_points_([0-9]{8})T([0-9]{2})Z[.]nc$")),
}

WINDOW_DEFINITIONS = {
    "full_day_00_23": {"start_hour": 0, "end_hour": 23, "expected_hours": 24},
    "trade_08_18": {"start_hour": 8, "end_hour": 18, "expected_hours": 11},
    "trade_08_19": {"start_hour": 8, "end_hour": 19, "expected_hours": 12},
}
WINDOW_ORDER = list(WINDOW_DEFINITIONS.keys())
WINDOW_EXPECTED_HOURS = {name: spec["expected_hours"] for name, spec in WINDOW_DEFINITIONS.items()}


def compute_expected_hours_for_window(local_date, aggregation_window, tz=LOCAL_TIMEZONE) -> int:
    """Return DST-aware expected local civil hours for a window."""
    window_name = str(aggregation_window)
    if window_name == "trade_08_18":
        return 11
    if window_name == "trade_08_19":
        return 12
    if window_name != "full_day_00_23":
        raise ValueError(f"Unknown aggregation_window: {aggregation_window}")

    local_day = pd.Timestamp(local_date).normalize()
    local_start = local_day.tz_localize(tz)
    local_end = (local_day + pd.Timedelta(days=1)).tz_localize(tz)
    delta = local_end.tz_convert("UTC") - local_start.tz_convert("UTC")
    return int(delta / pd.Timedelta(hours=1))


def assign_expected_hours_for_windows(frame: pd.DataFrame, date_col: str) -> pd.Series:
    """Vector-friendly wrapper for DST-aware expected window hours."""
    unique_pairs = frame[[date_col, "aggregation_window"]].drop_duplicates()
    expected_map = {
        (
            pd.Timestamp(row[date_col]).normalize(),
            str(row["aggregation_window"]),
        ): compute_expected_hours_for_window(row[date_col], row["aggregation_window"])
        for _, row in unique_pairs.iterrows()
    }
    keys = zip(pd.to_datetime(frame[date_col]).dt.normalize(), frame["aggregation_window"].astype(str))
    return pd.Series([expected_map[key] for key in keys], index=frame.index, dtype="int8")

DAILY_OUTPUT_HORIZONS = [1, 2]
WINDOW_OUTPUT_HORIZONS = [0, 1, 2]
MIN_HOURS_IN_TARGET = 12

LAT_LON_TOLERANCE = 0.001
LOG_EVERY = 100
N_STEPS = 67
SENTINEL_ABS_THRESHOLD = 1e6
PRECIP_TINY_NEGATIVE_TOLERANCE = -1e-5

for path, label in [
    (RAW_DATA_ROOT, "raw data root"),
    (AVD_LOKASJON, "Avd_lokasjon.xlsx"),
    (CFG_PATH, "cloud_points.cfg"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

for variant, (sub_folder, _ncvar, _suffix, _regex) in VARIANTS.items():
    folder = RAW_DATA_ROOT / sub_folder
    if not folder.exists():
        raise FileNotFoundError(f"Missing NetCDF folder for {variant}: {folder}")

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data root: {RAW_DATA_ROOT}")
print(f"Daily output: {OUTPUT_PATH}")
print(f"Window output: {WINDOW_OUTPUT_PATH}")
print(f"Horizon-hour audit output: {HOUR_AUDIT_OUTPUT_PATH}")
print(f"Realised reference (daily): {REALISED_DAILY_PATH} (exists={REALISED_DAILY_PATH.exists()})")
print(f"Realised reference (windows): {REALISED_WINDOWS_PATH} (exists={REALISED_WINDOWS_PATH.exists()})")


## Store mapping

The next step reads `Avd_lokasjon.xlsx` and `cloud_points.cfg`, verifies latitude/longitude alignment within `0.001` degrees, and stores the ordered `AvdelingID` list used to map NetCDF `x` indices to stores.


In [ ]:
def parse_cloud_points_cfg(cfg_path: Path) -> tuple[np.ndarray, np.ndarray]:
    text = cfg_path.read_text(encoding="utf-8", errors="replace")
    lat_match = re.search(r"^\s*latitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    lon_match = re.search(r"^\s*longitudeValues\s*=\s*([^\n]+)$", text, flags=re.MULTILINE)
    if not lat_match or not lon_match:
        raise ValueError(f"Could not locate latitudeValues / longitudeValues in {cfg_path}")
    lats = np.array([float(x.strip()) for x in lat_match.group(1).split(",") if x.strip()])
    lons = np.array([float(x.strip()) for x in lon_match.group(1).split(",") if x.strip()])
    return lats, lons


cfg_lats, cfg_lons = parse_cloud_points_cfg(CFG_PATH)
avd_df = pd.read_excel(AVD_LOKASJON)

def _find_col(frame, candidates):
    for c in candidates:
        if c in frame.columns:
            return c
    raise KeyError(f"None of {candidates} found in columns {list(frame.columns)}")

col_avd = _find_col(avd_df, ["AvdelingID", "Avdeling", "id"])
col_lat = _find_col(avd_df, ["Lat", "latitude", "Latitude", "lat"])
col_lon = _find_col(avd_df, ["Lon", "longitude", "Longitude", "lon"])

if len(avd_df) != len(cfg_lats):
    raise ValueError(f"Length mismatch: xlsx={len(avd_df)} vs cfg={len(cfg_lats)}")

excel_lats = avd_df[col_lat].to_numpy(dtype="float64")
excel_lons = avd_df[col_lon].to_numpy(dtype="float64")

lat_diff = np.abs(excel_lats - cfg_lats)
lon_diff = np.abs(excel_lons - cfg_lons)
lat_bad = int((lat_diff > LAT_LON_TOLERANCE).sum())
lon_bad = int((lon_diff > LAT_LON_TOLERANCE).sum())

if lat_bad > 0 or lon_bad > 0:
    raise AssertionError(
        f"Store-mapping validation failed: {lat_bad} latitudes / {lon_bad} longitudes exceed "
        f"tolerance {LAT_LON_TOLERANCE}. Order of xlsx and cfg may not match."
    )

AVDELING_BY_X = avd_df[col_avd].astype("int64").to_numpy()
N_STORES = len(AVDELING_BY_X)
assert N_STORES == 46

print(f"Stores validated: {N_STORES}")
print(f"Max lat diff: {float(lat_diff.max()):.6g}, max lon diff: {float(lon_diff.max()):.6g}")
print(f"AvdelingID range: {AVDELING_BY_X.min()}..{AVDELING_BY_X.max()}")


## File index

The file index parses `init_date` and `init_hour` from each deterministic MEPS filename by weather variant. It reports file counts, initialization-hour coverage, and date ranges before parsing.


In [ ]:
def build_file_index(variant: str) -> pd.DataFrame:
    sub, _ncvar, _suffix, regex = VARIANTS[variant]
    folder = RAW_DATA_ROOT / sub
    if not folder.exists():
        raise FileNotFoundError(f"Variant folder missing: {folder}")
    rows = []
    skipped = 0
    for path in folder.iterdir():
        if not path.is_file() or path.suffix != ".nc":
            continue
        m = regex.match(path.name)
        if m is None:
            skipped += 1
            continue
        ymd = m.group(1)
        hh = int(m.group(2))
        d = date(int(ymd[:4]), int(ymd[4:6]), int(ymd[6:8]))
        rows.append((variant, d, hh, str(path)))
    df = pd.DataFrame(rows, columns=["variant", "init_date", "init_hour", "filepath"])
    df = df.sort_values(["init_date", "init_hour"]).reset_index(drop=True)
    return df


variant_indexes = {}
for variant in VARIANTS:
    idx = build_file_index(variant)
    variant_indexes[variant] = idx
    n_files = len(idx)
    n_dates = idx["init_date"].nunique() if n_files > 0 else 0
    hour_dist = idx["init_hour"].value_counts().sort_index().to_dict()
    if n_files > 0:
        d_min, d_max = idx["init_date"].min(), idx["init_date"].max()
    else:
        d_min = d_max = None
    print(f"{variant:7s} files={n_files:>5,} dates={n_dates} range={d_min}..{d_max}  init_hours={hour_dist}")


## Variant parser

For each weather variant, the parser reads the forecast trajectory files, applies unit conversion, converts accumulated precipitation to hourly precipitation, and builds an hourly table keyed by forecast origin, lead time, store, and target date. Arrays are pre-allocated and trimmed after successful reads to reduce memory overhead.


In [ ]:
UNIT_CONVERTERS = {
    "temp":   lambda arr: arr.astype("float32") - 273.15,
    "wind":   lambda arr: arr.astype("float32"),
    "humid":  lambda arr: np.clip(arr.astype("float32") * 100.0, 0.0, 100.0),
    "cloud":  lambda arr: np.clip(arr.astype("float32") * 100.0, 0.0, 100.0),
}

variant_metadata_log = {}
precip_negative_warnings: list[str] = []


def log_variant_metadata_once(variant: str, ds: netCDF4.Dataset, ncvar: str) -> None:
    if variant in variant_metadata_log:
        return
    var = ds.variables[ncvar]
    attrs = {
        "units": getattr(var, "units", ""),
        "standard_name": getattr(var, "standard_name", ""),
        "long_name": getattr(var, "long_name", ""),
    }
    variant_metadata_log[variant] = attrs
    print(
        f"    {variant}: NetCDF metadata units={attrs['units']!r}, "
        f"standard_name={attrs['standard_name']!r}, long_name={attrs['long_name']!r}"
    )
    if variant == "precip":
        print(
            "    precip: precipitation_amount_acc is accumulated since forecast init; "
            "it will be differenced to hourly precipitation before aggregation."
        )


def mask_netcdf_array(raw) -> np.ndarray:
    if hasattr(raw, "filled"):
        raw = raw.filled(np.nan)
    arr = np.asarray(raw, dtype="float32").reshape(N_STEPS, -1)
    arr = np.where(np.abs(arr) > SENTINEL_ABS_THRESHOLD, np.nan, arr).astype("float32")
    if arr.shape[1] != N_STORES:
        raise ValueError(f"Unexpected payload shape: {arr.shape}; expected second dimension {N_STORES}")
    return arr


def accumulated_precip_to_hourly(acc: np.ndarray, source_path: str) -> np.ndarray:
    hourly = np.diff(acc, prepend=acc[0:1, :], axis=0).astype("float32")
    hourly[0, :] = 0.0
    tiny_negative = (hourly < 0.0) & (hourly >= PRECIP_TINY_NEGATIVE_TOLERANCE)
    impossible_negative = hourly < PRECIP_TINY_NEGATIVE_TOLERANCE
    n_tiny = int(np.count_nonzero(tiny_negative))
    n_impossible = int(np.count_nonzero(impossible_negative))
    if n_tiny:
        hourly[tiny_negative] = 0.0
    if n_impossible:
        min_negative = float(np.nanmin(hourly[impossible_negative]))
        precip_negative_warnings.append(
            (
                f"{source_path}: {n_impossible} impossible negative hourly precip values; "
                f"min={min_negative:.6g}; set to NaN"
            )
        )
        hourly[impossible_negative] = np.nan
    return hourly


def add_forecast_time_columns(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    df["origin_datetime_utc"] = pd.to_datetime(df["origin_datetime_utc"])
    df["target_datetime_utc"] = pd.to_datetime(df["target_datetime_utc"])
    utc = df["target_datetime_utc"].dt.tz_localize("UTC")
    local = utc.dt.tz_convert(LOCAL_TIMEZONE)
    df["target_datetime_local"] = local.dt.tz_localize(None)
    df["target_date"] = df["target_datetime_local"].dt.normalize()
    df["target_hour_local"] = local.dt.hour.astype("int8")
    df["target_hour_utc"] = df["target_datetime_utc"].dt.hour.astype("int8")
    df["target_date_utc"] = df["target_datetime_utc"].dt.normalize()
    df["horizon_days"] = (df["target_date"] - df["origin_date"]).dt.days.astype("int16")
    df["horizon_days_utc"] = (df["target_date_utc"] - df["origin_date"]).dt.days.astype("int16")
    return df


def parse_one_variant(variant: str) -> tuple[pd.DataFrame, list[str]]:
    idx = variant_indexes[variant]
    _sub, ncvar, _suffix, _regex = VARIANTS[variant]
    n_files = len(idx)
    if n_files == 0:
        return pd.DataFrame(), []

    total_rows = n_files * N_STEPS * N_STORES
    origin_dates = np.empty(total_rows, dtype="datetime64[D]")
    origin_hours = np.empty(total_rows, dtype="int8")
    origin_dt_arr = np.empty(total_rows, dtype="datetime64[h]")
    target_dt_arr = np.empty(total_rows, dtype="datetime64[h]")
    lead_hours_arr = np.empty(total_rows, dtype="int16")
    avd_arr = np.empty(total_rows, dtype="int64")
    val_arr = np.empty(total_rows, dtype="float32")

    corrupt_paths: list[str] = []
    cursor = 0
    t0 = time.time()

    for i, row in enumerate(idx.itertuples(index=False)):
        try:
            with netCDF4.Dataset(row.filepath, "r") as ds:
                log_variant_metadata_once(variant, ds, ncvar)
                raw = ds.variables[ncvar][:]
            arr = mask_netcdf_array(raw)
            if variant == "precip":
                values = accumulated_precip_to_hourly(arr, row.filepath)
            else:
                values = UNIT_CONVERTERS[variant](arr)
        except Exception as exc:
            corrupt_paths.append(f"{row.filepath} :: {type(exc).__name__}: {exc}")
            continue

        origin_dt = datetime(row.init_date.year, row.init_date.month, row.init_date.day, row.init_hour)
        origin_dt64 = np.datetime64(origin_dt, "h")
        for step in range(N_STEPS):
            target_dt = origin_dt + timedelta(hours=step)
            slot = slice(cursor, cursor + N_STORES)
            origin_dates[slot] = np.datetime64(row.init_date)
            origin_hours[slot] = row.init_hour
            origin_dt_arr[slot] = origin_dt64
            target_dt_arr[slot] = np.datetime64(target_dt, "h")
            lead_hours_arr[slot] = step
            avd_arr[slot] = AVDELING_BY_X
            val_arr[slot] = values[step, :]
            cursor += N_STORES

        if (i + 1) % LOG_EVERY == 0:
            elapsed = time.time() - t0
            rate = (i + 1) / max(elapsed, 1e-6)
            print(f"    {variant}: parsed {i + 1:>5,} / {n_files:,} files ({rate:.0f} files/s)")

    df = pd.DataFrame({
        "origin_date": pd.to_datetime(origin_dates[:cursor]),
        "origin_hour": origin_hours[:cursor],
        "origin_datetime_utc": pd.to_datetime(origin_dt_arr[:cursor]),
        "target_datetime_utc": pd.to_datetime(target_dt_arr[:cursor]),
        "lead_hour": lead_hours_arr[:cursor],
        "AvdelingID": avd_arr[:cursor],
        "value": val_arr[:cursor],
    })
    df["origin_date"] = df["origin_date"].dt.normalize()
    df = add_forecast_time_columns(df)
    return df, corrupt_paths


## Daily aggregation

Each parsed weather variant is aggregated to `(origin_date, origin_hour, target_date, AvdelingID)`. Temperature, wind, humidity, and cloud use mean/minimum/maximum summaries as specified, while precipitation is summed after accumulation-to-hourly conversion. `hours_in_target` records the number of lead hours contributing to each target day. Rows outside the retained horizon and coverage rules are removed at the end of this step.


In [ ]:
AGGREGATION_SPECS = {
    "temp":   {"mean": ("value", "mean"),  "min": ("value", "min"),  "max": ("value", "max")},
    "precip": {"sum":  ("value", "sum")},
    "wind":   {"mean": ("value", "mean"),  "max": ("value", "max")},
    "humid":  {"mean": ("value", "mean")},
    "cloud":  {"mean": ("value", "mean")},
}

KEY_COLS_DAILY = ["origin_date", "origin_hour", "target_date_utc", "horizon_days_utc", "AvdelingID"]
WINDOW_KEY_COLS = [
    "origin_date", "origin_hour", "origin_datetime_utc", "target_date", "horizon_days",
    "AvdelingID", "aggregation_window",
]
AUDIT_KEY_COLS = [
    "origin_date", "origin_hour", "origin_datetime_utc", "target_date", "horizon_days",
    "target_datetime_utc", "target_datetime_local", "target_hour_utc", "target_hour_local", "lead_hour",
]


def aggregate_variant_daily(hourly_df: pd.DataFrame, variant: str) -> pd.DataFrame:
    if hourly_df.empty:
        return pd.DataFrame()
    spec = AGGREGATION_SPECS[variant]
    agg_kwargs = {f"{variant}_fcst_{label}": pair for label, pair in spec.items()}
    daily = (
        hourly_df.groupby(KEY_COLS_DAILY, observed=True)
        .agg(hours_in_target=("value", "count"), **agg_kwargs)
        .reset_index()
    )
    mask = daily["horizon_days_utc"].isin(DAILY_OUTPUT_HORIZONS) & (
        daily["hours_in_target"] >= MIN_HOURS_IN_TARGET
    )
    return daily.loc[mask].reset_index(drop=True)


def aggregate_variant_windows(hourly_df: pd.DataFrame, variant: str) -> pd.DataFrame:
    if hourly_df.empty:
        return pd.DataFrame()
    spec = AGGREGATION_SPECS[variant]
    agg_kwargs = {f"{variant}_fcst_{label}": pair for label, pair in spec.items()}
    frames = []
    base_cols = [
        "origin_date", "origin_hour", "origin_datetime_utc", "target_date",
        "horizon_days", "AvdelingID", "target_datetime_utc", "target_hour_local",
        "target_hour_utc", "lead_hour", "value",
    ]
    horizon_mask = hourly_df["horizon_days"].isin(WINDOW_OUTPUT_HORIZONS)
    for window_name in WINDOW_ORDER:
        window = WINDOW_DEFINITIONS[window_name]
        mask = horizon_mask & hourly_df["target_hour_local"].between(
            window["start_hour"], window["end_hour"]
        )
        sub = hourly_df.loc[mask, base_cols]
        sub = sub.loc[sub["value"].notna()]
        if sub.empty:
            continue
        sub = sub.sort_values(WINDOW_KEY_COLS[:-1] + ["target_datetime_utc"])
        grouped = (
            sub.groupby(WINDOW_KEY_COLS[:-1], observed=True)
            .agg(
                **agg_kwargs,
                hours_in_window=("value", "size"),
                first_target_hour_local=("target_hour_local", "min"),
                last_target_hour_local=("target_hour_local", "max"),
                first_target_hour_utc=("target_hour_utc", "first"),
                last_target_hour_utc=("target_hour_utc", "last"),
                min_lead_hour=("lead_hour", "min"),
                max_lead_hour=("lead_hour", "max"),
            )
            .reset_index()
        )
        grouped["aggregation_window"] = window_name
        grouped["expected_hours_in_window"] = window["expected_hours"]
        frames.append(grouped)
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


def build_variant_hour_audit(hourly_df: pd.DataFrame, variant: str) -> pd.DataFrame:
    if hourly_df.empty:
        return pd.DataFrame()
    audit = (
        hourly_df.groupby(AUDIT_KEY_COLS, observed=True)
        .agg(stores_with_value=("value", "count"))
        .reset_index()
    )
    audit[f"{variant}_available"] = audit["stores_with_value"] > 0
    return audit.drop(columns=["stores_with_value"])


variant_daily = {}
variant_windows = {}
variant_hour_audits = {}
all_corrupt = []

for variant in VARIANTS:
    print(f"--- parsing {variant} ---")
    t0 = time.time()
    hourly_df, corrupt_paths = parse_one_variant(variant)
    print(f"    {variant}: hourly rows = {len(hourly_df):,}; corrupt files = {len(corrupt_paths)}")
    if corrupt_paths:
        all_corrupt.extend(corrupt_paths)
    daily = aggregate_variant_daily(hourly_df, variant)
    windows = aggregate_variant_windows(hourly_df, variant)
    hour_audit = build_variant_hour_audit(hourly_df, variant)
    print(
        f"    {variant}: daily rows = {len(daily):,}; window rows = {len(windows):,}; "
        f"hour-audit rows = {len(hour_audit):,}; took {(time.time()-t0)/60:.2f} min"
    )
    variant_daily[variant] = daily
    variant_windows[variant] = windows
    variant_hour_audits[variant] = hour_audit
    del hourly_df
    gc.collect()

print(f"\nTotal corrupt-file warnings: {len(all_corrupt)}")
if all_corrupt[:5]:
    for line in all_corrupt[:5]:
        print(f"  {line}")
print(f"Impossible negative precipitation warnings: {len(precip_negative_warnings)}")
if precip_negative_warnings[:5]:
    for line in precip_negative_warnings[:5]:
        print(f"  {line}")


## Variant merge

The five daily variant frames are inner-joined on forecast origin, target date, horizon, and `AvdelingID`. The merged output renames initialization fields to `origin_date` and `origin_hour` and validates uniqueness of the forecast panel keys.


In [ ]:
merged = None
for variant in VARIANTS:
    sub = variant_daily[variant].copy()
    value_cols = [c for c in sub.columns if c.startswith(f"{variant}_fcst_")]
    keep = KEY_COLS_DAILY + ["hours_in_target"] + value_cols
    sub = sub[keep].rename(columns={"hours_in_target": f"{variant}_hours_in_target"})
    if merged is None:
        merged = sub
    else:
        merged = merged.merge(sub, on=KEY_COLS_DAILY, how="inner")

print(f"Rows after inner join across variants for existing daily output: {len(merged):,}")

hours_cols = [f"{v}_hours_in_target" for v in VARIANTS]
merged["hours_in_target"] = merged[hours_cols].min(axis=1).astype("int8")
merged = merged.drop(columns=hours_cols)

merged = merged.rename(columns={
    "target_date_utc": "target_date",
    "horizon_days_utc": "horizon_days",
})
merged["origin_hour"] = merged["origin_hour"].astype("int8")
merged["horizon_days"] = merged["horizon_days"].astype("int8")
merged["AvdelingID"] = merged["AvdelingID"].astype("int64")

fcst_cols = [c for c in merged.columns if c.endswith(("_mean", "_min", "_max", "_sum"))]
for c in fcst_cols:
    merged[c] = merged[c].astype("float32")

ordered = [
    "origin_date", "origin_hour", "target_date", "horizon_days", "AvdelingID",
    "temp_fcst_mean", "temp_fcst_min", "temp_fcst_max",
    "precip_fcst_sum",
    "wind_fcst_mean", "wind_fcst_max",
    "humid_fcst_mean",
    "cloud_fcst_mean",
    "hours_in_target",
]
missing = [c for c in ordered if c not in merged.columns]
if missing:
    raise AssertionError(f"Merged frame is missing columns: {missing}")
merged = merged[ordered].sort_values(
    ["origin_date", "origin_hour", "target_date", "AvdelingID"]
).reset_index(drop=True)

key = ["origin_date", "origin_hour", "target_date", "AvdelingID"]
dup = int(merged.duplicated(key).sum())
if dup != 0:
    raise AssertionError(f"Final daily frame has {dup} duplicate keys on {key}")

print(f"Daily final frame: rows={len(merged):,}, cols={merged.shape[1]}")
print(f"Memory: {merged.memory_usage(deep=True).sum()/1_000_000:.1f} MB")
display(merged.head(5))


## Window forecasts and horizon-hour audit

This step builds `forecast_meps_daily_windows.parquet`, with one row per `(origin_date, origin_hour, target_date, horizon_days, AvdelingID, aggregation_window)` for `h=0`, `h=1`, and `h=2`. It also writes `forecast_meps_horizon_hour_audit.parquet`, which records target-hour availability for evaluating local-window coverage before downstream window selection.


In [ ]:
def select_forecast_window_columns(df: pd.DataFrame, variant: str) -> pd.DataFrame:
    value_cols = [c for c in df.columns if c.startswith(f"{variant}_fcst_")]
    keep = WINDOW_KEY_COLS + value_cols + [
        "hours_in_window", "first_target_hour_local", "last_target_hour_local",
        "first_target_hour_utc", "last_target_hour_utc", "min_lead_hour", "max_lead_hour",
    ]
    out = df[keep].rename(columns={
        "hours_in_window": f"{variant}_hours_in_window",
        "first_target_hour_local": f"{variant}_first_target_hour_local",
        "last_target_hour_local": f"{variant}_last_target_hour_local",
        "first_target_hour_utc": f"{variant}_first_target_hour_utc",
        "last_target_hour_utc": f"{variant}_last_target_hour_utc",
        "min_lead_hour": f"{variant}_min_lead_hour",
        "max_lead_hour": f"{variant}_max_lead_hour",
    })
    return out


forecast_windows = None
for variant in VARIANTS:
    sub = select_forecast_window_columns(variant_windows[variant], variant)
    if forecast_windows is None:
        forecast_windows = sub
    else:
        forecast_windows = forecast_windows.merge(sub, on=WINDOW_KEY_COLS, how="inner")

print(f"Rows after inner join across variants for window output: {len(forecast_windows):,}")

hours_cols = [f"{v}_hours_in_window" for v in VARIANTS]
first_local_cols = [f"{v}_first_target_hour_local" for v in VARIANTS]
last_local_cols = [f"{v}_last_target_hour_local" for v in VARIANTS]
first_utc_cols = [f"{v}_first_target_hour_utc" for v in VARIANTS]
last_utc_cols = [f"{v}_last_target_hour_utc" for v in VARIANTS]
min_lead_cols = [f"{v}_min_lead_hour" for v in VARIANTS]
max_lead_cols = [f"{v}_max_lead_hour" for v in VARIANTS]

forecast_windows["expected_hours_in_window"] = assign_expected_hours_for_windows(
    forecast_windows, "target_date"
)
forecast_windows["hours_in_window"] = forecast_windows[hours_cols].min(axis=1).astype("int8")
forecast_windows["coverage_share"] = np.minimum(
    forecast_windows["hours_in_window"].astype("float32") /
    forecast_windows["expected_hours_in_window"].astype("float32"),
    1.0,
).astype("float32")
forecast_windows["is_full_window"] = (
    forecast_windows["hours_in_window"] == forecast_windows["expected_hours_in_window"]
).astype("int8")
forecast_windows["is_partial_window"] = (1 - forecast_windows["is_full_window"]).astype("int8")

forecast_windows["first_target_hour_local"] = (
    forecast_windows[first_local_cols].max(axis=1, skipna=True).astype("int8")
)
forecast_windows["last_target_hour_local"] = (
    forecast_windows[last_local_cols].min(axis=1, skipna=True).astype("int8")
)
forecast_windows["first_target_hour_utc"] = (
    forecast_windows[first_utc_cols].max(axis=1, skipna=True).astype("int8")
)
forecast_windows["last_target_hour_utc"] = (
    forecast_windows[last_utc_cols].min(axis=1, skipna=True).astype("int8")
)
forecast_windows["min_lead_hour"] = (
    forecast_windows[min_lead_cols].max(axis=1, skipna=True).astype("int16")
)
forecast_windows["max_lead_hour"] = (
    forecast_windows[max_lead_cols].min(axis=1, skipna=True).astype("int16")
)

forecast_windows = forecast_windows.drop(
    columns=(
        hours_cols + first_local_cols + last_local_cols + first_utc_cols
        + last_utc_cols + min_lead_cols + max_lead_cols
    )
)

ordered_window_cols = [
    "origin_date", "origin_hour", "origin_datetime_utc", "target_date", "horizon_days",
    "AvdelingID", "aggregation_window",
    "temp_fcst_mean", "temp_fcst_min", "temp_fcst_max",
    "precip_fcst_sum",
    "wind_fcst_mean", "wind_fcst_max",
    "humid_fcst_mean",
    "cloud_fcst_mean",
    "hours_in_window", "expected_hours_in_window", "coverage_share",
    "is_partial_window", "is_full_window",
    "first_target_hour_local", "last_target_hour_local",
    "first_target_hour_utc", "last_target_hour_utc",
    "min_lead_hour", "max_lead_hour",
]
missing = [c for c in ordered_window_cols if c not in forecast_windows.columns]
if missing:
    raise AssertionError(f"Forecast window frame missing columns: {missing}")

forecast_windows = (
    forecast_windows[ordered_window_cols]
    .sort_values(WINDOW_KEY_COLS)
    .reset_index(drop=True)
)
forecast_windows["origin_date"] = pd.to_datetime(forecast_windows["origin_date"]).dt.normalize()
forecast_windows["target_date"] = pd.to_datetime(forecast_windows["target_date"]).dt.normalize()
forecast_windows["origin_datetime_utc"] = pd.to_datetime(forecast_windows["origin_datetime_utc"])
forecast_windows["origin_hour"] = forecast_windows["origin_hour"].astype("int8")
forecast_windows["horizon_days"] = forecast_windows["horizon_days"].astype("int8")
forecast_windows["AvdelingID"] = forecast_windows["AvdelingID"].astype("int64")
forecast_windows["aggregation_window"] = forecast_windows["aggregation_window"].astype("string")

for col in [
    "temp_fcst_mean", "temp_fcst_min", "temp_fcst_max", "precip_fcst_sum",
    "wind_fcst_mean", "wind_fcst_max", "humid_fcst_mean", "cloud_fcst_mean", "coverage_share",
]:
    forecast_windows[col] = forecast_windows[col].astype("float32")
for col in [
    "hours_in_window", "expected_hours_in_window", "is_partial_window", "is_full_window",
    "first_target_hour_local", "last_target_hour_local", "first_target_hour_utc", "last_target_hour_utc",
]:
    forecast_windows[col] = forecast_windows[col].astype("int8")
for col in ["min_lead_hour", "max_lead_hour"]:
    forecast_windows[col] = forecast_windows[col].astype("int16")

dup_window = int(forecast_windows.duplicated(WINDOW_KEY_COLS).sum())
if dup_window != 0:
    raise AssertionError(
        f"Forecast window frame has {dup_window} duplicate keys on {WINDOW_KEY_COLS}"
    )

horizon_hour_audit = None
for variant in VARIANTS:
    sub = variant_hour_audits[variant]
    if horizon_hour_audit is None:
        horizon_hour_audit = sub
    else:
        horizon_hour_audit = horizon_hour_audit.merge(sub, on=AUDIT_KEY_COLS, how="outer")

available_cols = [f"{v}_available" for v in VARIANTS]
horizon_hour_audit[available_cols] = horizon_hour_audit[available_cols].fillna(False)
horizon_hour_audit["available"] = horizon_hour_audit[available_cols].all(axis=1)
horizon_hour_audit = (
    horizon_hour_audit[AUDIT_KEY_COLS + ["available"]]
    .sort_values(AUDIT_KEY_COLS)
    .reset_index(drop=True)
)
for col in ["origin_date", "target_date"]:
    horizon_hour_audit[col] = pd.to_datetime(horizon_hour_audit[col]).dt.normalize()
for col in ["origin_datetime_utc", "target_datetime_utc", "target_datetime_local"]:
    horizon_hour_audit[col] = pd.to_datetime(horizon_hour_audit[col])
for col in ["origin_hour", "horizon_days", "target_hour_utc", "target_hour_local"]:
    horizon_hour_audit[col] = horizon_hour_audit[col].astype("int8")
horizon_hour_audit["lead_hour"] = horizon_hour_audit["lead_hour"].astype("int16")

print(f"Forecast window rows: {len(forecast_windows):,}")
print(f"Horizon-hour audit rows: {len(horizon_hour_audit):,}")
display(
    forecast_windows.groupby(["horizon_days", "aggregation_window"], observed=True)
    .agg(
        rows=("target_date", "size"),
        mean_coverage=("coverage_share", "mean"),
        full_share=("is_full_window", "mean"),
    )
    .reset_index()
    .round(4)
)


## Parquet checkpoint

The daily, window, and horizon-hour audit parquets are written before downstream diagnostics so that parsed forecast outputs are preserved if a later validation step fails.


In [ ]:
checkpoint = merged.copy()
checkpoint["origin_date"] = checkpoint["origin_date"].dt.normalize()
checkpoint["target_date"] = checkpoint["target_date"].dt.normalize()
checkpoint.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")

forecast_windows_checkpoint = forecast_windows.copy()
forecast_windows_checkpoint["origin_date"] = forecast_windows_checkpoint["origin_date"].dt.normalize()
forecast_windows_checkpoint["target_date"] = forecast_windows_checkpoint["target_date"].dt.normalize()
forecast_windows_checkpoint.to_parquet(
    WINDOW_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy"
)

horizon_hour_audit_checkpoint = horizon_hour_audit.copy()
horizon_hour_audit_checkpoint["origin_date"] = horizon_hour_audit_checkpoint["origin_date"].dt.normalize()
horizon_hour_audit_checkpoint["target_date"] = horizon_hour_audit_checkpoint["target_date"].dt.normalize()
horizon_hour_audit_checkpoint.to_parquet(
    HOUR_AUDIT_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy"
)

print(
    f"Daily checkpoint written: {OUTPUT_PATH} "
    f"({OUTPUT_PATH.stat().st_size/1_000_000:.2f} MB, {len(checkpoint):,} rows)"
)
print(
    f"Window checkpoint written: {WINDOW_OUTPUT_PATH} "
    f"({WINDOW_OUTPUT_PATH.stat().st_size/1_000_000:.2f} MB, "
    f"{len(forecast_windows_checkpoint):,} rows)"
)
print(
    f"Hour-audit checkpoint written: {HOUR_AUDIT_OUTPUT_PATH} "
    f"({HOUR_AUDIT_OUTPUT_PATH.stat().st_size/1_000_000:.2f} MB, "
    f"{len(horizon_hour_audit_checkpoint):,} rows)"
)
del checkpoint, forecast_windows_checkpoint, horizon_hour_audit_checkpoint
gc.collect()


## Validation against realised output

This validation compares a sample of h=1 `temp_fcst_mean` values with h=1 target-day `temp_obs_mean` from notebook 01a. The check is a parser sanity test against realised MET Nordic Analysis proxy values, not a model result.


In [ ]:
if not REALISED_DAILY_PATH.exists():
    print(f"Realised reference not found at {REALISED_DAILY_PATH}; skipping validation.")
else:
    realised = pd.read_parquet(REALISED_DAILY_PATH)
    realised["date"] = pd.to_datetime(realised["date"]).dt.normalize()
    realised = realised.rename(columns={"date": "target_date"})

    h1 = merged.loc[
        merged["horizon_days"] == 1,
        [
            "target_date", "AvdelingID", "temp_fcst_mean", "precip_fcst_sum",
            "wind_fcst_mean", "humid_fcst_mean", "cloud_fcst_mean",
        ],
    ].copy()
    h1["target_date"] = h1["target_date"].dt.normalize()

    joined = h1.merge(
        realised[["target_date", "AvdelingID", "temp_obs_mean", "precip_obs_sum",
                  "wind_obs_mean", "humid_obs_mean", "cloud_obs_mean"]],
        on=["target_date", "AvdelingID"], how="inner",
    )

    print(f"Joined rows for h=1 validation: {len(joined):,}")
    pairs = [
        ("temp_fcst_mean", "temp_obs_mean", "deg C"),
        ("precip_fcst_sum", "precip_obs_sum", "mm"),
        ("wind_fcst_mean", "wind_obs_mean", "m/s"),
        ("humid_fcst_mean", "humid_obs_mean", "percent"),
        ("cloud_fcst_mean", "cloud_obs_mean", "percent"),
    ]
    rows = []
    for fcst_col, real_col, unit in pairs:
        sub = joined[[fcst_col, real_col]].dropna()
        if len(sub) < 10:
            rows.append({
                "var": fcst_col,
                "n": len(sub),
                "corr": float("nan"),
                "mean_diff": float("nan"),
                "rmse": float("nan"),
                "unit": unit,
            })
            continue
        corr = float(sub[fcst_col].corr(sub[real_col]))
        diff = (sub[fcst_col] - sub[real_col]).astype("float64")
        rows.append({
            "var": fcst_col,
            "n": int(len(sub)),
            "corr_pearson": corr,
            "mean_diff_fcst_minus_real": float(diff.mean()),
            "rmse": float(np.sqrt((diff ** 2).mean())),
            "unit": unit,
        })
    validation_df = pd.DataFrame(rows)
    display(validation_df.round(4))

    if rows[0].get("corr_pearson", 0) < 0.85:
        print(f"\nWARNING: temperature h=1 correlation = {rows[0]['corr_pearson']:.3f} (< 0.85).")
    else:
        print(f"\nOK: temperature h=1 correlation = {rows[0]['corr_pearson']:.3f}")
    del realised, joined, h1
    gc.collect()


## Sanity checks

The following checks summarize physical ranges, missing values, duplicate keys, precipitation non-negativity, and sentinel-value guards in the deterministic forecast outputs.


In [ ]:
bounds = {
    "temp_fcst_mean":  (-40.0, 40.0),
    "temp_fcst_min":   (-50.0, 40.0),
    "temp_fcst_max":   (-40.0, 50.0),
    "precip_fcst_sum": (0.0, 300.0),
    "wind_fcst_mean":  (0.0, 50.0),
    "wind_fcst_max":   (0.0, 80.0),
    "humid_fcst_mean": (0.0, 100.0),
    "cloud_fcst_mean": (0.0, 100.0),
}

SENTINEL = 1e30
for frame_name, frame in [("daily", merged), ("windows", forecast_windows)]:
    _sentinel_mask = frame[list(bounds.keys())].abs().gt(SENTINEL).any(axis=1)
    _n_sentinel = int(_sentinel_mask.sum())
    if _n_sentinel > 0:
        raise AssertionError(
            f"{frame_name}: {_n_sentinel} rows contain sentinel-magnitude forecast "
            f"values (>{SENTINEL:.0e})."
        )
    for col, (lo, hi) in bounds.items():
        vmin = float(frame[col].min())
        vmax = float(frame[col].max())
        if vmin < lo or vmax > hi:
            raise AssertionError(
                f"{frame_name}: {col} out of range: min={vmin}, max={vmax}, "
                f"expected [{lo}, {hi}]"
            )

print(f"Daily rows: {len(merged):,}")
print(
    f"Daily origin date range: {merged['origin_date'].min().date()} -> "
    f"{merged['origin_date'].max().date()}"
)
print(
    f"Daily target date range: {merged['target_date'].min().date()} -> "
    f"{merged['target_date'].max().date()}"
)
print(f"Window rows: {len(forecast_windows):,}")
print(
    f"Window target date range: {forecast_windows['target_date'].min().date()} -> "
    f"{forecast_windows['target_date'].max().date()}"
)
print(f"Window duplicate keys: {int(forecast_windows.duplicated(WINDOW_KEY_COLS).sum())}")

print("\nDaily init-hour distribution:")
display(
    merged["origin_hour"]
    .value_counts()
    .sort_index()
    .rename_axis("origin_hour")
    .reset_index(name="rows")
)
print("\nDaily horizon-days distribution:")
display(
    merged["horizon_days"]
    .value_counts()
    .sort_index()
    .rename_axis("horizon_days")
    .reset_index(name="rows")
)
print("\nWindow row counts by horizon/window:")
display(
    forecast_windows.groupby(["horizon_days", "aggregation_window"], observed=True)
    .size()
    .rename("rows")
    .reset_index()
)
print("\nWindow coverage summary:")
display(
    forecast_windows.groupby(["horizon_days", "aggregation_window"], observed=True)
    .agg(
        rows=("target_date", "size"),
        min_hours=("hours_in_window", "min"),
        median_hours=("hours_in_window", "median"),
        max_hours=("hours_in_window", "max"),
        mean_coverage=("coverage_share", "mean"),
        full_share=("is_full_window", "mean"),
    )
    .reset_index()
    .round(4)
)
print("\nValue distributions, daily output:")
display(merged[list(bounds.keys())].describe(percentiles=[0.25, 0.5, 0.75]).round(3).T)


## Horizon-hour coverage audit

This coverage-only summary evaluates whether candidate local-time trading windows are sufficiently observed at h=2. It informs downstream window choice but does not define the final modelling specification by itself.


In [ ]:
h2_full = forecast_windows.loc[
    (forecast_windows["horizon_days"] == 2) &
    (forecast_windows["aggregation_window"] == "full_day_00_23")
].copy()

print("h=2 last_target_hour_local distribution, full_day_00_23:")
display(
    h2_full["last_target_hour_local"]
    .value_counts(normalize=False)
    .sort_index()
    .rename_axis("last_target_hour_local")
    .reset_index(name="rows")
)
print("h=2 last_target_hour_utc distribution, full_day_00_23:")
display(
    h2_full["last_target_hour_utc"]
    .value_counts(normalize=False)
    .sort_index()
    .rename_axis("last_target_hour_utc")
    .reset_index(name="rows")
)

coverage_by_window = (
    forecast_windows.groupby(["horizon_days", "aggregation_window"], observed=True)
    .agg(
        rows=("target_date", "size"),
        mean_coverage=("coverage_share", "mean"),
        full_share=("is_full_window", "mean"),
        median_last_local=("last_target_hour_local", "median"),
    )
    .reset_index()
)
print("Coverage by horizon_days and aggregation_window:")
display(coverage_by_window.round(4))

keep_rows = []
for threshold in [0.50, 0.75, 0.90, 1.00]:
    tmp = (
        forecast_windows.assign(keep=forecast_windows["coverage_share"] >= threshold)
        .groupby(["horizon_days", "aggregation_window"], observed=True)["keep"]
        .mean()
        .reset_index()
    )
    tmp["coverage_threshold"] = threshold
    keep_rows.append(tmp)
keep_rates = pd.concat(keep_rows, ignore_index=True)[
    ["horizon_days", "aggregation_window", "coverage_threshold", "keep"]
].rename(columns={"keep": "keep_rate"})
print("Keep-rates by coverage threshold:")
display(keep_rates.round(4))

h2_0818 = forecast_windows.loc[
    (forecast_windows["horizon_days"] == 2) &
    (forecast_windows["aggregation_window"] == "trade_08_18"),
    "is_full_window",
].mean()
h2_0819 = forecast_windows.loc[
    (forecast_windows["horizon_days"] == 2) &
    (forecast_windows["aggregation_window"] == "trade_08_19"),
    "is_full_window",
].mean()
most_common_last_local = h2_full["last_target_hour_local"].mode()
most_common_last_local = (
    int(most_common_last_local.iloc[0]) if not most_common_last_local.empty else None
)
partial_gap = float(h2_0818 - h2_0819)

print("Explicit h=2 coverage answers:")
print(f"1. Most common last available local hour for h=2: {most_common_last_local}")
print(f"2. Share of h=2 rows fully covering trade_08_18: {h2_0818:.4f}")
print(f"3. Share of h=2 rows fully covering trade_08_19: {h2_0819:.4f}")
print(f"4. Full-coverage gap trade_08_18 minus trade_08_19: {partial_gap:.4f}")
if partial_gap >= 0.05:
    print(
        "5. Coverage-only recommendation: trade_08_18 is safer than "
        "trade_08_19 for later ML screening."
    )
else:
    print(
        "5. Coverage-only recommendation: both trade_08_18 and trade_08_19 "
        "have similar h=2 coverage; compare empirically downstream."
    )
print("Note: this is not a final modelling choice; full_day_00_23 remains the benchmark.")


## Diagnostic plots

The optional diagnostics visualize forecast time series, forecast-versus-realised scatter, and error distributions by horizon. They support parser inspection and are not thesis result figures.


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

rng = np.random.default_rng(seed=42)
sample_avd = int(rng.choice(AVDELING_BY_X))

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("MEPS deterministic forecast diagnostics")

sub_ts = merged.loc[merged["AvdelingID"] == sample_avd]
for h, color in [(1, "tab:blue"), (2, "tab:orange")]:
    sub_h = sub_ts.loc[sub_ts["horizon_days"] == h].sort_values("target_date")
    axes[0, 0].plot(sub_h["target_date"], sub_h["temp_fcst_mean"], lw=0.6, color=color, label=f"h={h}")
axes[0, 0].set_title(f"temp_fcst_mean at AvdelingID={sample_avd}")
axes[0, 0].set_ylabel("deg C")
axes[0, 0].legend()

if REALISED_DAILY_PATH.exists():
    realised = pd.read_parquet(REALISED_DAILY_PATH)[
        ["date", "AvdelingID", "temp_obs_mean"]
    ].rename(columns={"date": "target_date"})
    realised["target_date"] = pd.to_datetime(realised["target_date"]).dt.normalize()
    plot_df = merged[["target_date", "AvdelingID", "horizon_days", "temp_fcst_mean"]].copy()
    plot_df["target_date"] = plot_df["target_date"].dt.normalize()
    plot_df = plot_df.merge(realised, on=["target_date", "AvdelingID"], how="inner")
    sample_for_plot = plot_df.sample(min(len(plot_df), 5000), random_state=0)
    for h, color in [(1, "tab:blue"), (2, "tab:orange"), (3, "tab:green")]:
        sub_h = sample_for_plot.loc[sample_for_plot["horizon_days"] == h]
        if len(sub_h) > 0:
            axes[0, 1].scatter(sub_h["temp_obs_mean"], sub_h["temp_fcst_mean"],
                               s=4, alpha=0.3, color=color, label=f"h={h}")
    axes[0, 1].plot([-30, 30], [-30, 30], "k--", lw=0.5)
    axes[0, 1].set_xlabel("realised temp_obs_mean (deg C)")
    axes[0, 1].set_ylabel("forecast temp_fcst_mean (deg C)")
    axes[0, 1].set_title("Forecast vs realised temperature")
    axes[0, 1].legend()

    plot_df["err"] = plot_df["temp_fcst_mean"] - plot_df["temp_obs_mean"]
    for h, color in [(1, "tab:blue"), (2, "tab:orange"), (3, "tab:green")]:
        sub_h = plot_df.loc[plot_df["horizon_days"] == h, "err"]
        if len(sub_h) > 0:
            axes[1, 0].hist(sub_h, bins=80, alpha=0.4, color=color, label=f"h={h}")
    axes[1, 0].set_title("Forecast error (fcst - realised), temperature")
    axes[1, 0].set_xlabel("deg C")
    axes[1, 0].legend()

    box_data = [
        plot_df.loc[plot_df["horizon_days"] == h, "err"].dropna()
        for h in [1, 2, 3]
        if (plot_df["horizon_days"] == h).sum() > 0
    ]
    box_labels = [
        f"h={h}" for h in [1, 2, 3]
        if (plot_df["horizon_days"] == h).sum() > 0
    ]
    axes[1, 1].boxplot(box_data, labels=box_labels, showfliers=False)
    axes[1, 1].set_title("Forecast error boxplot by horizon")
    axes[1, 1].set_ylabel("deg C")
    del realised, plot_df

plt.tight_layout()
plt.show()
plt.close(fig)


## Final save and summary

The final deterministic forecast panels and audit table are written after validation, followed by a compact summary of row counts, horizons, and output locations.


In [ ]:
merged.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
forecast_windows.to_parquet(WINDOW_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")
horizon_hour_audit.to_parquet(HOUR_AUDIT_OUTPUT_PATH, index=False, engine="pyarrow", compression="snappy")

print(f"Final daily parquet written: {OUTPUT_PATH}")
print(f"Final window parquet written: {WINDOW_OUTPUT_PATH}")
print(f"Final horizon-hour audit parquet written: {HOUR_AUDIT_OUTPUT_PATH}")
print(f"Daily size on disk: {OUTPUT_PATH.stat().st_size/1_000_000:.2f} MB")
print(f"Window size on disk: {WINDOW_OUTPUT_PATH.stat().st_size/1_000_000:.2f} MB")
print(f"Audit size on disk: {HOUR_AUDIT_OUTPUT_PATH.stat().st_size/1_000_000:.2f} MB")
print(f"Daily rows: {len(merged):,}")
print(f"Window rows: {len(forecast_windows):,}")
print(f"Hour-audit rows: {len(horizon_hour_audit):,}")

print("\nDaily schema:")
for col, dt in merged.dtypes.items():
    print(f"  {col:25s} {dt}")
print("\nWindow schema:")
for col, dt in forecast_windows.dtypes.items():
    print(f"  {col:25s} {dt}")
print("\nHour-audit schema:")
for col, dt in horizon_hour_audit.dtypes.items():
    print(f"  {col:25s} {dt}")

print("\nDaily distributions:")
display(
    merged[
        ["temp_fcst_mean", "temp_fcst_min", "temp_fcst_max", "precip_fcst_sum",
         "wind_fcst_mean", "wind_fcst_max", "humid_fcst_mean", "cloud_fcst_mean"]
    ].describe(percentiles=[0.25, 0.5, 0.75]).round(3).T
)
print("\nDaily head 5:")
display(merged.head(5))
print(f"\nCorrupt files: {len(all_corrupt)}")
if all_corrupt:
    for line in all_corrupt[:10]:
        print(f"  {line}")
print(f"Impossible negative precipitation warnings: {len(precip_negative_warnings)}")
if precip_negative_warnings:
    for line in precip_negative_warnings[:10]:
        print(f"  {line}")


## Forecast-window validation

After parsing, this validation checks the forecast-window parquet, compares the local full-day windows for h=1 and h=2 against the existing UTC-day forecast output, and optionally joins realised windows when notebook 01a outputs are available.


In [ ]:
forecast_daily_check = pd.read_parquet(OUTPUT_PATH)
forecast_windows_check = pd.read_parquet(WINDOW_OUTPUT_PATH)
hour_audit_check = pd.read_parquet(HOUR_AUDIT_OUTPUT_PATH)

for frame, cols in [
    (forecast_daily_check, ["origin_date", "target_date"]),
    (forecast_windows_check, ["origin_date", "origin_datetime_utc", "target_date"]),
    (
        hour_audit_check,
        [
            "origin_date", "origin_datetime_utc", "target_date",
            "target_datetime_utc", "target_datetime_local",
        ],
    ),
]:
    for col in cols:
        frame[col] = pd.to_datetime(frame[col])

print("Forecast window row counts by aggregation_window and horizon_days:")
display(
    forecast_windows_check.groupby(["horizon_days", "aggregation_window"], observed=True)
    .size()
    .rename("rows")
    .reset_index()
)

expected_horizons = set(WINDOW_OUTPUT_HORIZONS)
actual_horizons = set(forecast_windows_check["horizon_days"].dropna().astype(int).unique())
if not expected_horizons.issubset(actual_horizons):
    raise AssertionError(
        f"Missing expected forecast-window horizons: "
        f"{sorted(expected_horizons - actual_horizons)}"
    )

print("Coverage summary by horizon/window:")
display(
    forecast_windows_check.groupby(["horizon_days", "aggregation_window"], observed=True)
    .agg(
        rows=("target_date", "size"),
        mean_coverage=("coverage_share", "mean"),
        full_share=("is_full_window", "mean"),
        min_last_local=("last_target_hour_local", "min"),
        median_last_local=("last_target_hour_local", "median"),
        max_last_local=("last_target_hour_local", "max"),
    )
    .reset_index()
    .round(4)
)

print("h=2 last available local-hour distribution:")
display(
    forecast_windows_check.loc[
        (forecast_windows_check["horizon_days"] == 2) &
        (forecast_windows_check["aggregation_window"] == "full_day_00_23"),
        "last_target_hour_local",
    ].value_counts().sort_index().rename_axis("last_target_hour_local").reset_index(name="rows")
)

key_window = [
    "origin_date", "origin_hour", "target_date", "horizon_days",
    "AvdelingID", "aggregation_window",
]
dup_window = int(forecast_windows_check.duplicated(key_window).sum())
if dup_window:
    raise AssertionError(f"Forecast window duplicate keys found: {dup_window}")

value_cols = [
    "temp_fcst_mean", "temp_fcst_min", "temp_fcst_max", "precip_fcst_sum",
    "wind_fcst_mean", "wind_fcst_max", "humid_fcst_mean", "cloud_fcst_mean",
]
if forecast_windows_check[value_cols].abs().gt(SENTINEL_ABS_THRESHOLD).any().any():
    raise AssertionError("Sentinel-magnitude values found in forecast window output.")
if forecast_windows_check["precip_fcst_sum"].min() < 0:
    raise AssertionError("Negative precipitation found in forecast window output.")
for col in ["humid_fcst_mean", "cloud_fcst_mean"]:
    if forecast_windows_check[col].min() < 0 or forecast_windows_check[col].max() > 100:
        raise AssertionError(f"{col} outside [0, 100].")

full_day = forecast_windows_check.loc[
    forecast_windows_check["aggregation_window"] == "full_day_00_23",
    ["origin_date", "origin_hour", "target_date", "horizon_days", "AvdelingID"] +
    ["temp_fcst_mean", "precip_fcst_sum", "wind_fcst_mean", "humid_fcst_mean", "cloud_fcst_mean"],
]
compare = full_day.merge(
    forecast_daily_check[
        ["origin_date", "origin_hour", "target_date", "horizon_days", "AvdelingID"]
        + [
            "temp_fcst_mean", "precip_fcst_sum", "wind_fcst_mean",
            "humid_fcst_mean", "cloud_fcst_mean",
        ]
    ],
    on=["origin_date", "origin_hour", "target_date", "horizon_days", "AvdelingID"],
    how="inner",
    suffixes=("_local_full_day", "_utc_daily"),
)
compare = compare.loc[compare["horizon_days"].isin([1, 2])]
print(f"Joined rows for local full-day h=1/h=2 vs UTC daily comparison: {len(compare):,}")
rows = []
for col in [
    "temp_fcst_mean", "precip_fcst_sum", "wind_fcst_mean",
    "humid_fcst_mean", "cloud_fcst_mean",
]:
    left = f"{col}_local_full_day"
    right = f"{col}_utc_daily"
    sub = compare[[left, right]].dropna()
    diff = sub[left].astype("float64") - sub[right].astype("float64")
    rows.append({
        "variable": col,
        "n": int(len(sub)),
        "corr": float(sub[left].corr(sub[right])) if len(sub) > 1 else np.nan,
        "mean_diff_local_minus_utc": float(diff.mean()) if len(sub) else np.nan,
        "mean_abs_diff": float(diff.abs().mean()) if len(sub) else np.nan,
    })
display(pd.DataFrame(rows).round(5))


## Forecast-versus-realised window diagnostics

This diagnostic compares deterministic MEPS forecasts with realised MET Nordic Analysis proxy values by local-time aggregation window and horizon. The resulting forecast-error summaries are validation and calibration diagnostics, not operational ML features.


In [ ]:
if not REALISED_WINDOWS_PATH.exists():
    print(
        f"Realised window parquet not found at {REALISED_WINDOWS_PATH}; "
        "run notebook 01a first to enable window-level forecast validation."
    )
else:
    realised_windows_check = pd.read_parquet(REALISED_WINDOWS_PATH)
    realised_windows_check["date"] = pd.to_datetime(realised_windows_check["date"]).dt.normalize()
    forecast_windows_for_validation = forecast_windows_check.copy()
    forecast_windows_for_validation["target_date"] = pd.to_datetime(
        forecast_windows_for_validation["target_date"]
    ).dt.normalize()

    joined = forecast_windows_for_validation.merge(
        realised_windows_check[[
            "date", "AvdelingID", "aggregation_window",
            "temp_obs_mean", "precip_obs_sum", "wind_obs_mean", "humid_obs_mean", "cloud_obs_mean",
        ]],
        left_on=["target_date", "AvdelingID", "aggregation_window"],
        right_on=["date", "AvdelingID", "aggregation_window"],
        how="inner",
    )
    print(f"Joined forecast-realised window rows: {len(joined):,}")

    pairs = [
        ("temp_fcst_mean", "temp_obs_mean"),
        ("precip_fcst_sum", "precip_obs_sum"),
        ("wind_fcst_mean", "wind_obs_mean"),
        ("humid_fcst_mean", "humid_obs_mean"),
        ("cloud_fcst_mean", "cloud_obs_mean"),
    ]
    metric_rows = []
    for (horizon, window_name), group in joined.groupby(
        ["horizon_days", "aggregation_window"], observed=True
    ):
        row = {
            "horizon_days": int(horizon),
            "aggregation_window": window_name,
            "n": int(len(group)),
        }
        for fcst_col, obs_col in pairs:
            sub = group[[fcst_col, obs_col]].dropna()
            diff = sub[fcst_col].astype("float64") - sub[obs_col].astype("float64")
            prefix = fcst_col.replace("_fcst", "")
            row[f"{prefix}_corr"] = float(sub[fcst_col].corr(sub[obs_col])) if len(sub) > 1 else np.nan
            row[f"{prefix}_mean_diff"] = float(diff.mean()) if len(sub) else np.nan
            row[f"{prefix}_rmse"] = float(np.sqrt((diff ** 2).mean())) if len(sub) else np.nan
            row[f"{prefix}_mae"] = float(diff.abs().mean()) if len(sub) else np.nan

        wet_threshold = 0.1
        wet = group[["precip_fcst_sum", "precip_obs_sum"]].dropna()
        obs_wet = wet["precip_obs_sum"] >= wet_threshold
        fcst_wet = wet["precip_fcst_sum"] >= wet_threshold
        row["obs_wet_share"] = float(obs_wet.mean()) if len(wet) else np.nan
        row["fcst_wet_share"] = float(fcst_wet.mean()) if len(wet) else np.nan
        row["hit_rate"] = (
            float((fcst_wet & obs_wet).sum() / max(obs_wet.sum(), 1))
            if len(wet) else np.nan
        )
        row["miss_rate"] = (
            float((~fcst_wet & obs_wet).sum() / max(obs_wet.sum(), 1))
            if len(wet) else np.nan
        )
        dry_count = int((~obs_wet).sum())
        row["false_alarm_rate"] = (
            float((fcst_wet & ~obs_wet).sum() / max(dry_count, 1))
            if len(wet) else np.nan
        )
        row["accuracy"] = float((fcst_wet == obs_wet).mean()) if len(wet) else np.nan
        metric_rows.append(row)

    forecast_vs_realised_metrics = pd.DataFrame(metric_rows).sort_values(
        ["horizon_days", "aggregation_window"]
    )
    display(forecast_vs_realised_metrics.round(4))
    del joined
    gc.collect()


## Manual run notes

Run notebook 01a first if `realised_weather_daily_windows.parquet` is missing, then run this notebook from the top through validation. Confirm that `forecast_meps_daily.parquet`, `forecast_meps_daily_windows.parquet`, and `forecast_meps_horizon_hour_audit.parquet` are written under `data/processed/`. Inspect the h=2 last-available local-hour distribution, full-window coverage for `trade_08_18` and `trade_08_19`, and keep-rates at 0.50, 0.75, 0.90, and 1.00 coverage.

The parser processes one forecast variable at a time and discards hourly frames after aggregation. `precipitation_amount_acc` is masked for sentinels before differencing; small negative artefacts are clipped to zero, while materially negative values are set to NaN and logged. Later documentation updates should record actual row counts, file sizes, coverage results, validation metrics, and the final coverage-based window recommendation. Realised weather and forecast weather remain separate, and forecast errors are validation/calibration diagnostics rather than operational ML features.
